# 06 — Faithfulness Judge  
## Qwen/Qwen2.5-7B-Instruct · Kaggle T4 · Tüm 8 ablasyon

RAGAS-style LLM-as-judge faithfulness scoring — tüm 8 ablasyon × 122 soru = **976 satır**.  
API key gerektirmez. Tek T4'te 4-bit quantization ile çalışır.

### Gerekli Kaggle dataset'leri

Bu notebook'u çalıştırmadan önce **iki dataset** eklenmiş olmalı:

1. **`turkish-legal-rag-system`** (zaten var)  
   → `data/corpus/mevzuat_full_normalized.jsonl` içermeli

2. **`rag-per-examples`** (yeni, sen oluştur)  
   → Aşağıdaki klasör yapısını zip'le, Kaggle'a dataset olarak yükle:
   ```
   rag-per-examples/
     A1/per_example.jsonl
     A2/per_example.jsonl
     A3/per_example.jsonl
     A4/per_example.jsonl
     A5/per_example.jsonl
     A5a/per_example.jsonl
     A5b/per_example.jsonl
     A5c/per_example.jsonl
   ```
   Kaynak: `nlp_term_project/outputs/results_v2/<ABLATION>/per_example.jsonl`

### Süre tahmini
- Model yükleme: ~5 dk  
- 976 sorgu × ~8 s/sorgu (T4, 4-bit) ≈ **2.5 saat**  
- Kaggle session limiti 9 saat → sorunsuz tamamlanır

### Çıktılar (`/kaggle/working/judge_output/`)
- `A1/with_judge.jsonl` … `A5c/with_judge.jsonl`  
- `judge_summary.json` (her ablasyon için mean faithfulness)

In [ ]:
# ── 0. Paket kurulumu ──────────────────────────────────────────────────────
%%capture
!pip install -q bitsandbytes>=0.41.0 accelerate transformers>=4.40.0

In [ ]:
# ── 1. Path konfigürasyonu ─────────────────────────────────────────────────
import subprocess
from pathlib import Path

# Dataset 1: corpus (turkish-legal-rag-finetune içinde)
FINETUNE_DS = Path('/kaggle/input/datasets/hasanemreusta/turkish-legal-rag-finetune')

# Corpus dosyasını bul
_candidates = [
    FINETUNE_DS / 'mevzuat_full_normalized.jsonl',
    FINETUNE_DS / 'data/corpus/mevzuat_full_normalized.jsonl',
    FINETUNE_DS / 'corpus/mevzuat_full_normalized.jsonl',
]
CORPUS_FILE = next((p for p in _candidates if p.exists()), None)

if CORPUS_FILE is None:
    print('‼ CORPUS BULUNAMADI — gerçek path aranıyor...')
    out = subprocess.run(
        ['find', str(FINETUNE_DS), '-name', 'mevzuat_full_normalized.jsonl'],
        capture_output=True, text=True
    ).stdout.strip()
    print(out or '(bulunamadı)')
else:
    print('✓ Corpus:', CORPUS_FILE)

# Dataset 2: per_example.jsonl dosyaları
PER_EX_DS = Path('/kaggle/input/datasets/hasanemreusta/rag-per-examples')

# Çıktı dizini
OUT_DIR = Path('/kaggle/working/judge_output')
OUT_DIR.mkdir(exist_ok=True)

ABLATIONS = ['A1', 'A2', 'A3', 'A4', 'A5', 'A5a', 'A5b', 'A5c']

# ── Per-example kontrol ────────────────────────────────────────────────────
print()
all_ok = True
for abl in ABLATIONS:
    p = PER_EX_DS / abl / 'per_example.jsonl'
    status = '✓' if p.exists() else '✗ EKSİK'
    print(f'  {abl}: {status}')
    if not p.exists():
        all_ok = False

if not all_ok:
    print('\n‼ Bazı per_example.jsonl eksik')
else:
    print('\n✓ Tüm dosyalar hazır.')

In [ ]:
# ── 2. Corpus yükle ────────────────────────────────────────────────────────
import json, re, time, gc
import torch

print('Corpus yükleniyor...')
corpus = {}
with CORPUS_FILE.open(encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        doc = json.loads(line)
        if doc.get('doc_id'):
            corpus[doc['doc_id']] = doc
print(f'Corpus yüklendi: {len(corpus):,} madde')

In [ ]:
# ── 3. Model yükle (4-bit) ─────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'{MODEL_ID} yükleniyor...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.eval()
print(f'Model yüklendi ({time.time()-t0:.0f}s). VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# ── 4. Judge fonksiyonları ─────────────────────────────────────────────────
PROMPT = """Sen bir hukuki RAG sisteminin \"faithfulness\" (kaynak-bağlılık) hakemisin.
Aşağıda bir soru, modelin cevabı ve cevabın dayanması gereken KAYNAK MADDELER var.

Görevin:
1. Cevabı atomik iddialara (kısa, doğrulanabilir cümleler) ayır.
2. Her iddiayı, SADECE kaynak maddelerdeki bilgiye dayanarak değerlendir:
   - SUPPORTED: iddia tamamen kaynaktan çıkarılabilir
   - PARTIAL: kısmen destekleniyor ama eksik/uydurma kısımlar var
   - UNSUPPORTED: kaynaklardan çıkarılamıyor (hallucination)
3. faithfulness = SUPPORTED_sayisi / toplam_iddia (PARTIAL=0.5 say)

SORU: {question}

CEVAP:
\"\"\"
{answer}
\"\"\"

KAYNAK MADDELER:
\"\"\"
{context}
\"\"\"

Yalnızca aşağıdaki formatta geçerli JSON döndür, başka hiçbir şey yazma:
{{
  \"claims\": [
    {{\"claim\": \"...\", \"verdict\": \"SUPPORTED\", \"reason\": \"...\"}}
  ],
  \"faithfulness\": 0.0
}}"""


def build_context(doc_ids, max_chars=3000):
    parts, used = [], 0
    for doc_id in (doc_ids or []):
        doc = corpus.get(doc_id)
        if not doc:
            continue
        head = f"[{doc.get('kanun_short', '?')} m.{doc.get('madde_no', '?')}]"
        block = head + '\n' + (doc.get('text') or '')
        if used + len(block) > max_chars:
            break
        parts.append(block)
        used += len(block) + 2
    return '\n\n'.join(parts)


def parse_json(text):
    text = (text or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    s, e = text.find('{'), text.rfind('}')
    if s == -1 or e == -1:
        return None
    try:
        return json.loads(text[s:e + 1])
    except json.JSONDecodeError:
        return None


@torch.inference_mode()
def judge_one(question, answer, context):
    prompt_text = PROMPT.format(question=question, answer=answer, context=context)
    messages = [{'role': 'user', 'content': prompt_text}]
    formatted = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors='pt', truncation=True, max_length=3500).to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False,
        temperature=1.0,          # do_sample=False → greedy, temperature ignored
        pad_token_id=tokenizer.eos_token_id,
    )
    new_ids = outputs[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_ids, skip_special_tokens=True)

    parsed = parse_json(raw)
    if not parsed or 'claims' not in parsed:
        return {'faithfulness': None, 'claims': [], 'error': 'parse_failed', 'raw': raw[:300]}

    claims = parsed.get('claims') or []
    if not claims:
        return {'faithfulness': None, 'claims': [], 'error': 'no_claims'}

    score = 0.0
    for c in claims:
        v = (c.get('verdict') or '').upper()
        if v == 'SUPPORTED':
            score += 1.0
        elif v == 'PARTIAL':
            score += 0.5
    return {'faithfulness': score / len(claims), 'claims': claims}


print('Judge fonksiyonları hazır.')

In [ ]:
# ── 5. Smoke test (1 satır) ────────────────────────────────────────────────
smoke_file = PER_EX_DS / 'A1' / 'per_example.jsonl'
smoke_row = json.loads(smoke_file.read_text(encoding='utf-8').splitlines()[0])
smoke_ctx = build_context(smoke_row.get('retrieved_doc_ids', []))

print(f'Soru: {smoke_row["question"][:80]}')
print(f'Context uzunluğu: {len(smoke_ctx)} karakter')

t0 = time.time()
smoke_result = judge_one(smoke_row['question'], smoke_row['answer'], smoke_ctx)
print(f'\nSonuç: faithfulness={smoke_result.get("faithfulness")} | error={smoke_result.get("error")} | {time.time()-t0:.1f}s')
print(f'Claims ({len(smoke_result.get("claims", []))}):')
for c in smoke_result.get('claims', [])[:3]:
    print(f'  [{c.get("verdict")}] {c.get("claim", "")[:70]}')

In [ ]:
# ── 6. Tüm 8 ablasyon — tam koşum (resume destekli) ───────────────────────
from statistics import mean as _mean

all_summaries = {}
grand_start = time.time()

for abl in ABLATIONS:
    in_path = PER_EX_DS / abl / 'per_example.jsonl'
    if not in_path.exists():
        print(f'[{abl}] ATLA — per_example.jsonl bulunamadı')
        continue

    (OUT_DIR / abl).mkdir(exist_ok=True)
    out_path = OUT_DIR / abl / 'with_judge.jsonl'
    tmp_path = out_path.with_suffix('.jsonl.tmp')

    rows = [
        json.loads(l)
        for l in in_path.read_text(encoding='utf-8').splitlines()
        if l.strip()
    ]

    # Resume: daha önce tamamlanmışsa atla
    if out_path.exists():
        done = [json.loads(l) for l in out_path.read_text(encoding='utf-8').splitlines() if l.strip()]
        scored = [r['faithfulness'] for r in done if r.get('faithfulness') is not None]
        mean_f = _mean(scored) if scored else None
        mf_str = f'{mean_f:.4f}' if mean_f is not None else 'N/A'
        print(f'[{abl}] ATLA (zaten tamamlandı) — scored={len(scored)}/{len(done)} mean_faith={mf_str}')
        all_summaries[abl] = {'n_rows': len(done), 'n_scored': len(scored),
                               'n_failed': len(done)-len(scored), 'mean_faithfulness': mean_f}
        continue

    print(f'\n{"="*55}', flush=True)
    print(f'  {abl}  ({len(rows)} satır)', flush=True)
    print(f'{"="*55}', flush=True)

    scored_vals, n_failed = [], 0
    t_abl = time.time()

    with tmp_path.open('w', encoding='utf-8') as fout:
        for i, row in enumerate(rows):
            q   = row.get('question', '')
            a   = row.get('answer', '')
            ctx = build_context(row.get('retrieved_doc_ids', []))

            if not a or not ctx:
                result = {'faithfulness': None, 'claims': [], 'error': 'missing'}
                n_failed += 1
            else:
                result = judge_one(q, a, ctx)
                if result.get('error'):
                    n_failed += 1

            merged = dict(row)
            merged['faithfulness']  = result['faithfulness']
            merged['judge_claims']  = result.get('claims', [])
            if result.get('error'):
                merged['judge_error'] = result['error']

            fout.write(json.dumps(merged, ensure_ascii=False) + '\n')
            fout.flush()

            if merged['faithfulness'] is not None:
                scored_vals.append(merged['faithfulness'])

            if (i + 1) % 10 == 0:
                avg     = _mean(scored_vals) if scored_vals else float('nan')
                elapsed = time.time() - t_abl
                eta     = (elapsed / (i + 1)) * (len(rows) - i - 1)
                print(
                    f'  [{i+1:3d}/{len(rows)}] '
                    f'avg_faith={avg:.3f} '
                    f'failed={n_failed} '
                    f'elapsed={elapsed/60:.1f}dk '
                    f'ETA={eta/60:.1f}dk',
                    flush=True
                )

    tmp_path.replace(out_path)

    mean_f = _mean(scored_vals) if scored_vals else None
    mf_str = f'{mean_f:.4f}' if mean_f is not None else 'N/A'
    summary = {
        'n_rows':            len(rows),
        'n_scored':          len(scored_vals),
        'n_failed':          n_failed,
        'mean_faithfulness': mean_f,
    }
    all_summaries[abl] = summary

    print(
        f'  ✓ {abl}: scored={len(scored_vals)}/{len(rows)} '
        f'mean_faith={mf_str} '
        f'({(time.time()-t_abl)/60:.1f} dk)',
        flush=True
    )

    gc.collect()
    torch.cuda.empty_cache()

total_min = (time.time() - grand_start) / 60
print(f'\nToplam süre: {total_min:.1f} dakika', flush=True)

# judge_summary.json kaydet
(OUT_DIR / 'judge_summary.json').write_text(
    json.dumps(all_summaries, indent=2, ensure_ascii=False)
)
print('\njudge_summary.json:')
print(json.dumps(all_summaries, indent=2, ensure_ascii=False))

In [ ]:
# ── 7. Özet tablo ──────────────────────────────────────────────────────────
print(f'{'Ablasyon':<8} {'Scored':>8} {'Failed':>8} {'Mean Faith':>12}')
print('-' * 42)
for abl, s in all_summaries.items():
    mf = f"{s['mean_faithfulness']:.4f}" if s['mean_faithfulness'] is not None else 'N/A'
    print(f"{abl:<8} {s['n_scored']:>8}/{s['n_rows']:<4} {s['n_failed']:>8} {mf:>12}")

print()
print('Çıktılar: /kaggle/working/judge_output/')
print('Output sekmesinden zip olarak indir.')